Deep Learning Embeddings Roadmap: Co-occurrence ->
Word2Vec -> GloVe
Phase 1: Co-occurrence Matrix
Goal: Understand the raw statistics behind word embeddings.

1. Build a small corpus manually (10–20 sentences).
2. Construct the co-occurrence matrix X, where X_ij = count of word j in context of word
   i.
3. Normalize rows to get probabilities P(j | i).
4. Observe that words appearing in similar contexts have similar row vectors.
   Phase 2: Word2Vec (Skip-gram / CBOW)
   Goal: See how predictive models turn raw counts into dense embeddings.
5. Implement skip-gram on your small corpus.
6. Optional: Add negative sampling to approximate softmax.
7. Train for a few epochs and extract learned embeddings.
8. Compute cosine similarity between words.
9. Compare embeddings with raw co-occurrence vectors to see compression into lower
   dimensions.
   Phase 3: Analogy and Similarity Tasks
   Goal: See emergent semantics from embeddings.
10. Test word relationships: e.g., "king - man + woman".
11. Observe how skip-gram captures semantic relationships unlike raw co-occurrence
    matrices.
    Phase 4: GloVe (Global Vectors)
    Goal: Bridge count-based and predictive embeddings.
12. Understand GloVe objective: J = sum_ij f(X_ij) (w_i^T w_j + b_i + b_j - log X_ij)^2.
13. Use your co-occurrence matrix from Phase 1.
14. Implement simple matrix factorization (SVD) on log(X).
15. Compare embeddings to skip-gram embeddings.
16. Observe that both capture semantic similarity via different mechanisms (predictive
    vs count-based).
    Phase 5: Visualization & Intuition Check
17. Reduce embeddings to 2D using PCA/t-SNE.
18. Plot words like "cat", "dog", "king", "queen".
19. Compare raw co-occurrence vectors, skip-gram embeddings, and GloVe embeddings.
20. Insight: embeddings encode context similarity, which produces semantics.
    Outcome After These Experiments

- Understand math behind embeddings (raw counts -> predictive -> global).
- See embeddings emerge visually.
- Understand why Word2Vec and GloVe work and how they relate.
- Ready for contextual embeddings (ELMo, BERT) and transformers with full
  understanding.


way to do the nlp


- do the cooccurance matrix -- with 1 if present if not 0 ###not gona do it due to it is not feasible practically

- the co occurance matrix with the PMI just construct with small sample

  - two ways if less then zero then zero
  - positive pmi

- dimension reduction using svd


In [6]:
import numpy as np
import nltk
from scipy.sparse.linalg import svds
import sys
nltk.download('brown')

from nltk.corpus import brown



[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\sp710\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [7]:
# constants

WINDOW_SIZE: int=3

k:int= 50

In [8]:
sentences=brown.sents()[:100]

corpus = [sentence[i:i+WINDOW_SIZE] for sentence in sentences for i in range(len(sentence)-WINDOW_SIZE+1)]

print(corpus)

[['The', 'Fulton', 'County'], ['Fulton', 'County', 'Grand'], ['County', 'Grand', 'Jury'], ['Grand', 'Jury', 'said'], ['Jury', 'said', 'Friday'], ['said', 'Friday', 'an'], ['Friday', 'an', 'investigation'], ['an', 'investigation', 'of'], ['investigation', 'of', "Atlanta's"], ['of', "Atlanta's", 'recent'], ["Atlanta's", 'recent', 'primary'], ['recent', 'primary', 'election'], ['primary', 'election', 'produced'], ['election', 'produced', '``'], ['produced', '``', 'no'], ['``', 'no', 'evidence'], ['no', 'evidence', "''"], ['evidence', "''", 'that'], ["''", 'that', 'any'], ['that', 'any', 'irregularities'], ['any', 'irregularities', 'took'], ['irregularities', 'took', 'place'], ['took', 'place', '.'], ['The', 'jury', 'further'], ['jury', 'further', 'said'], ['further', 'said', 'in'], ['said', 'in', 'term-end'], ['in', 'term-end', 'presentments'], ['term-end', 'presentments', 'that'], ['presentments', 'that', 'the'], ['that', 'the', 'City'], ['the', 'City', 'Executive'], ['City', 'Executive'

In [9]:
def PPMI(cooccurance,N):
    '''
        PMI(w, c) = log p(c|w)
        p(c)
        = log |  count(w, c) ∗ N    |
              |-------------------  |
              |count(c) ∗ count(w)  |
    '''
    countWords = np.sum(cooccurance, axis=1)
    
    print(countWords)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        expected = np.outer(countWords, countWords) / N
        pmi = np.log10(cooccurance / expected)    
        pmi[np.isnan(pmi)] = 0
        pmi[pmi < 0] = 0
        
    cooccurance[:, :] = pmi
    return cooccurance



In [10]:

words=sorted(set(word for sentence in sentences for word in sentence))

word2id = {w: i for i, w in enumerate(words)}
id2word = {i: w for w, i in word2id.items()}

#calculating the coocurance matrix

cooccurance=np.zeros((len(words),len(words)),dtype=float)

for window in corpus:
    for i, word in enumerate(window):
        w_idx = word2id[word]

        for j, neighbor in enumerate(window):
            if i != j:
                n_idx = word2id[neighbor]
                cooccurance[w_idx, n_idx] += 1
np.set_printoptions(threshold=sys.maxsize)

print("Co-occurrence sum:", len(corpus),np.sum(cooccurance))

Co-occurrence sum: 2069 12414.0


In [ ]:
# calculating the ppmi mat
ppmi_mat=PPMI(cooccurance,len(words))

print(ppmi_mat)


In [12]:
### doing the svd (singular value decomposition on it)

def svd(matrix):
    
    U, S, Vt = svds(matrix,k=k)

    return [U[:, ::-1], S[::-1], Vt[::-1, :]]
    

svd_ppmi=svd(ppmi_mat)

for i in svd_ppmi:
    print(np.shape(i))
    
### and then doing the approximation

Word_embeding = svd_ppmi[0] @ np.diag(svd_ppmi[1])

print(f"the size of the new Embeding is {np.shape(Word_embeding)}")

(860, 50)
(50,)
(50, 860)
the size of the new Embeding is (860, 50)


In [13]:
# print(Word_embeding)

print(words)
print(np.dot(Word_embeding[word2id['Fulton']],Word_embeding[word2id['County']].T))

['$10', '$100', '$3', '$30', '$4', '$50', "''", '(', ')', ',', '--', '.', '1', '1,119', '13', '13th', '18', '1913', '1923', '1937', '1958', '1961', '1962', '2', '29-5', '402', '637', '71', '74', '8', '87-31', ':', 'A', 'After', 'Aj', 'Ala.', 'Allen', 'Alpharetta', 'As', 'Ask', 'Association', 'Atlanta', "Atlanta's", 'Attorneys', 'Aug.', 'Austin', 'Authority', 'B.', 'Bar', 'Barber', 'Before', 'Being', 'Bellwood', 'Berry', 'Blue', 'Board', 'Bowden', 'Bush', 'But', 'Byrd', "Byrd's", 'Caldwell', "Caldwell's", 'Callan', 'Carey', 'Chairman', 'Cheshire', 'City', 'Colquitt', 'Commerce', "Commissioner's", 'Committee', 'Congress', 'Constitution', 'Construction', 'County', 'Court', 'D.', "Daniel's", 'Davis', 'Democratic', 'Department', "Department's", 'Despite', 'Dorsey', 'During', 'Durwood', 'E.', 'Education', 'Everything', 'Executive', 'Failure', 'Felix', 'Five', 'Four', 'Friday', 'Fulton', 'GOP', 'Gainesville', 'Garland', 'George', 'Georgia', "Georgia's", 'Gov.', 'Grady', 'Grand', 'Griffin', 'H

# Continious Bag of word

In [14]:
# datasets
np_corpus = np.array(corpus)
print(np.shape(np_corpus))

X_train = np_corpus[:int(0.7*np.shape(np_corpus)[0]), 0 : WINDOW_SIZE-1]
Y_train = np_corpus[:int(0.7*np.shape(np_corpus)[0]) , WINDOW_SIZE-1:WINDOW_SIZE]

X_test = np_corpus[int(0.7*np.shape(np_corpus)[0])  : , 0 : WINDOW_SIZE-1]
Y_test = np_corpus[int(0.7*np.shape(np_corpus)[0])  : , WINDOW_SIZE-1:WINDOW_SIZE]


print(np.shape(X_train),np.shape(Y_train))
print(np.shape(X_test),np.shape(Y_test))


(2069, 3)
(1448, 2) (1448, 1)
(621, 2) (621, 1)


In [15]:
# constants

L=3 #is the no if layer

MAX_ITR=100 

LEARNING_RATE=0.5 #η 

NO_OF_INPUT=[(WINDOW_SIZE-1)*len(words) , (WINDOW_SIZE-1) * k , len(words)] # mem friendly input h1,h2 output

NO_OF_OUTPUT=len(words)

BATCH_SIZE=10

In [17]:
# Continuous bag of word exeqution


def initialize_parameters(layer_dims, method="xavier"):
    """
    Initializes weights and biases for a fully connected neural network.
    
    Parameters:
    -----------
    layer_dims : list of int
        Sizes of each layer in the network. Example: [784, 128, 64, 10]
    method : str
        Initialization method: "xavier" or "he"
    
    Returns:
    --------
    weights : list of np.ndarray
        Weight matrices for each layer
    biases : list of np.ndarray
        Bias vectors for each layer
    """
    weights = [] #(784,128) , (128,64) ,(64,10)
    biases = [] #(764 x 1 , 128 x 1 , 64 x 1 , 10 x 1)
    
    for i in range(len(layer_dims)-1):
        n_in = layer_dims[i]
        n_out = layer_dims[i+1]
        
        if method == "xavier":
            W = np.random.randn(n_in, n_out) * np.sqrt(1.0 / n_in)
        elif method == "he":
            W = np.random.randn(n_in,n_out) * np.sqrt(2.0 / n_in)
        else:
            raise ValueError("Invalid method. Use 'xavier' or 'he'.")
        
        b = np.zeros((1, n_out))
        
        weights.append(W)
        biases.append(b)
    
    return weights, biases

weight,bias=initialize_parameters(NO_OF_INPUT)

print(weight[0].shape)
print(weight[1].shape)
print(bias[0].shape)
print(bias[1].shape)




(1720, 100)
(100, 860)
(1, 100)
(1, 860)


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def loss_fn(y_pred, y_true):
    m = y_true.shape[0]
    return -np.sum(y_true * np.log(y_pred + 1e-9)) / m

def eigen(output):
    ans=np.eye(10)[output]
    return ans

def update_parameters(grads_w, grads_b):
    for i in range(L):
        weight[i] -= LEARNING_RATE * grads_w[i]
        bias[i] -= LEARNING_RATE * grads_b[i]


def forwardpropogation(input): #(10,784)
    activation,preactivation=[input],[]
    #(10,784)  ()
    
    for i in range(L):
        
        preactivation.append(np.dot(activation[-1],weight[i]) + bias[i])  # w0(784,128) , w1(128,64) ,w2(64,10) a0=(10,784) a1(10,128) a2(10,64)

        #preac (128,10) (64,10) (10,10)
        if i == L - 1:
            A = softmax(preactivation[i])  # output layer 
        else:
            A = sigmoid(preactivation[i])  # hidden layers (128,10)

        activation.append(A)
   
    return activation,preactivation # a0=(10,784) a1(10,128) a2(10,64) a3(10,10) #preac (128,10) (64,10) (10,10)


def backpropogation(activation, preactivation, y):
    m = y.shape[0]
    dz = activation[-1] - y  # last layer
    grad_w = [None] * L
    grad_b = [None] * L

    for i in reversed(range(L)):
        A_prev = activation[i]
        grad_w[i] = np.dot(A_prev.T, dz) / m
        grad_b[i] = np.sum(dz, axis=0, keepdims=True) / m

        if i > 0:  # for hidden layers
            dA_prev = np.dot(dz, weight[i].T)  # fix transpose here
            sigmoid_Z = 1 / (1 + np.exp(-preactivation[i-1]))  # no transpose here
            dz = dA_prev * (sigmoid_Z * (1 - sigmoid_Z))
    return grad_w, grad_b

In [21]:
######################################################## MINI_BATCH_GD################################ 
for epoch in range(MAX_ITR):
    idx=np.arange(len(X_train))
    np.random.shuffle(idx)

    X_train=X_train[idx] #(60000,28,28)
    Y_train=Y_train[idx] #(60000)


    for i in range(0,len(X_train),BATCH_SIZE):
        X_batch=X_train[i : i+BATCH_SIZE]
        X_batch = X_batch.reshape(X_batch.shape[0], -1) #(batch_size,784)

        Y_batch=eigen(Y_train[i:i+BATCH_SIZE]) #(10,10)
        activation,preactivation=forwardpropogation(X_batch) ##a()  p()
        
        loss=loss_fn(activation[-1],Y_batch)
        print(loss)
        dW,db=backpropogation(activation,preactivation,Y_batch)

IndexError: arrays used as indices must be of integer (or boolean) type